# 01b · Train the segmentation formulation

Same data, split, backbone and checkpoint rule as `01_train.ipynb`, but the generator predicts
the three map classes (background, road, water) with a softmax head and is trained with
soft Dice plus class-weighted cross-entropy instead of road-weighted L1. Labels are derived
from the RGB targets on the fly, so no mask files are needed. The map tile is recovered by
argmax and a palette lookup.

The thesis runs used `ARCH = "unet"`, no adversarial term and horizontal flips, with seeds
1907096 and 2024. Edit `CFG`, then Run All. Final numbers come from `02_evaluate.ipynb`,
which recognises these runs by their `seg_` prefix.

## 1. Imports

In [ ]:
import csv, json, math, time, random
from pathlib import Path

import numpy as np
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt

from sat2map import models, data, training, evaluation as ev, runs

print("TensorFlow", tf.__version__)
for g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(g, True)
print("GPUs:", tf.config.list_physical_devices("GPU"))

from sat2map import segmentation as seg

## 2. Configuration

In [ ]:
CFG = dict(
    # architecture
    ARCH        = "unet",       # unet | unet_bottleneck | unet_mid | resnet9 | attn_unet
    POS_EMBED   = True,
    NUM_HEADS   = 4,
    FF_DIM      = 1024,

    # data
    IMG_SIZE    = 256,
    SUBSET      = 0,            # 0 = all training files
    AUG         = "hflip",      # hflip (thesis) | d4 (all eight square symmetries)

    # loss
    DICE_WEIGHT = 1.0,
    CE_WEIGHT   = 1.0,
    ADV_WEIGHT  = 0.0,          # 0.0 in the thesis runs, discriminator unused
    CLASS_W     = (0.2, 1.0, 0.5),   # background, road, water
    LOSS_THRESH = 0.98,         # road class: every channel at least this
    WATER_DELTA = 0.10,         # water class: blue minus red at least this

    # training
    BASE_EPOCHS = 30,
    BATCH       = 4,
    LR          = 2e-4,
    BETA_1      = 0.5,
    SEED        = 2024,         # thesis runs: 1907096 and 2024

    # checkpoint selection
    EVAL_SIZE   = 256,
    BUFFER_PX   = 2,
    GEN_THRESH  = 0.50,         # road probability
    TAR_THRESH  = 0.90,         # target road mask, same as the image formulation
    EVAL_EVERY  = 2,
    TRAIN_EVAL_N = 100,
)

DATA_ROOT = "../data/Dataset"   # folder holding train/ val/ test/
OUT_ROOT  = "../runs"           # each run writes to OUT_ROOT/<run name>/

random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); tf.random.set_seed(CFG["SEED"])
SIZE = CFG["IMG_SIZE"]

## 3. Data

In [ ]:
root = Path(DATA_ROOT)
if not (root / "train").is_dir():
    raise SystemExit("No train/ folder under %s. Fix DATA_ROOT in section 2." % root)

ALL_TRAIN  = data.file_list(root, "train")
VAL_FILES  = data.file_list(root, "val")
TEST_FILES = data.file_list(root, "test")

if CFG["SUBSET"]:
    TRAIN_FILES = ALL_TRAIN[:CFG["SUBSET"]]
    CFG["EPOCHS"] = int(round(CFG["BASE_EPOCHS"] * len(ALL_TRAIN) / len(TRAIN_FILES)))
else:
    TRAIN_FILES = ALL_TRAIN
    CFG["EPOCHS"] = CFG["BASE_EPOCHS"]
CFG["N_TRAIN"] = len(TRAIN_FILES)
CFG["RUN_NAME"] = runs.seg_run_name(CFG["ARCH"], SIZE, CFG["DICE_WEIGHT"], CFG["CE_WEIGHT"],
                                    CFG["ADV_WEIGHT"], CFG["AUG"], CFG["N_TRAIN"], CFG["SEED"])
OUT = Path(OUT_ROOT) / CFG["RUN_NAME"]
OUT.mkdir(parents=True, exist_ok=True)

train_ds      = data.make_ds(TRAIN_FILES, SIZE, CFG["BATCH"], training=True,
                             seed=CFG["SEED"], aug=CFG["AUG"])
val_ds        = data.make_ds(VAL_FILES, SIZE)
test_ds       = data.make_ds(TEST_FILES, SIZE)
train_eval_ds = data.make_ds(TRAIN_FILES[:CFG["TRAIN_EVAL_N"]], SIZE)

print("train %d   val %d   test %d   | %d epochs" % (len(TRAIN_FILES), len(VAL_FILES),
                                                    len(TEST_FILES), CFG["EPOCHS"]))
print("run:", CFG["RUN_NAME"])

Sanity check on the derived labels: road should come out near 3% of pixels, and water above
zero. If water is exactly zero the palette assumption is wrong and the third class does nothing.

In [ ]:
counts = np.zeros(models.N_CLASS)
for _, t in train_ds.unbatch().take(200):
    lab = seg.labels_from_target(t[None], CFG["LOSS_THRESH"], CFG["WATER_DELTA"]).numpy().ravel()
    counts += np.bincount(lab, minlength=models.N_CLASS)
frac = counts / counts.sum()
print("class fractions over 200 training targets:  background %.4f  road %.4f  water %.4f"
      % (frac[models.BG], frac[models.ROAD], frac[models.WATER]))

## 4. Models

In [ ]:
generator = models.make_generator(CFG["ARCH"], SIZE, "seg", CFG["NUM_HEADS"],
                                  CFG["FF_DIM"], CFG["POS_EMBED"])
discriminator = models.build_discriminator(SIZE) if CFG["ADV_WEIGHT"] > 0 else None
print("%s generator (seg head): %.2fM params, output %s"
      % (CFG["ARCH"], generator.count_params() / 1e6, generator.output_shape))
print("adversarial term", "on" if discriminator is not None else "off")

g_opt = tf.keras.optimizers.Adam(CFG["LR"], beta_1=CFG["BETA_1"])
d_opt = tf.keras.optimizers.Adam(CFG["LR"], beta_1=CFG["BETA_1"])
train_step = seg.make_seg_train_step(generator, discriminator, g_opt, d_opt,
                                     CFG["CLASS_W"], CFG["DICE_WEIGHT"], CFG["CE_WEIGHT"],
                                     CFG["ADV_WEIGHT"], CFG["LOSS_THRESH"], CFG["WATER_DELTA"])

def quick_eval(ds):
    return seg.quick_eval_seg(generator, ds, CFG["EVAL_SIZE"], CFG["GEN_THRESH"],
                              CFG["TAR_THRESH"], CFG["BUFFER_PX"])

## 5. Train

In [ ]:
log_path, best_path = OUT / "history.csv", OUT / "generator_best.weights.h5"
best_iou, best_epoch, history = -1.0, 0, []
t0 = time.time()

for epoch in range(1, CFG["EPOCHS"] + 1):
    tot, n = np.zeros(5), 0
    for src, tar in train_ds:
        tot += [float(v) for v in train_step(src, tar)]
        n += 1
    tot /= n

    row = dict(epoch=epoch, g_total=tot[0], g_dice=tot[1], g_ce=tot[2], g_adv=tot[3],
               d_loss=tot[4], train_strict=np.nan, train_buf=np.nan,
               val_strict=np.nan, val_buf=np.nan, minutes=(time.time() - t0) / 60.0)

    if epoch % CFG["EVAL_EVERY"] == 0 or epoch == CFG["EPOCHS"]:
        tr_s, tr_b = quick_eval(train_eval_ds)
        va_s, va_b = quick_eval(val_ds)
        row.update(train_strict=tr_s, train_buf=tr_b, val_strict=va_s, val_buf=va_b)
        marker = ""
        if va_b > best_iou:
            best_iou, best_epoch = va_b, epoch
            generator.save_weights(str(best_path))
            marker = "  <- best"
        print("epoch %3d  dice %.4f  ce %.4f | train %.4f  val %.4f  gap %.4f%s"
              % (epoch, tot[1], tot[2], tr_b, va_b, tr_b - va_b, marker))
    else:
        print("epoch %3d  dice %.4f  ce %.4f" % (epoch, tot[1], tot[2]))

    history.append(row)
    with open(log_path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(history[0].keys()))
        w.writeheader(); w.writerows(history)

print("\nbest val buffered IoU %.4f at epoch %d, after %.1f minutes"
      % (best_iou, best_epoch, (time.time() - t0) / 60.0))

## 6. Curves

In [ ]:
h = {k: np.array([r[k] for r in history], float) for k in history[0]}
ev_ = ~np.isnan(h["val_buf"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(h["epoch"], h["g_dice"], label="dice")
ax[0].plot(h["epoch"], h["g_ce"], label="cross-entropy")
if CFG["ADV_WEIGHT"] > 0:
    ax[0].plot(h["epoch"], h["g_adv"], label="adversarial")
    ax[0].plot(h["epoch"], h["d_loss"], label="discriminator")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss"); ax[0].legend(); ax[0].set_title("losses")
ax[1].plot(h["epoch"][ev_], h["train_buf"][ev_], label="train buffered")
ax[1].plot(h["epoch"][ev_], h["val_buf"][ev_], label="val buffered")
ax[1].plot(h["epoch"][ev_], h["val_strict"][ev_], "--", label="val strict")
ax[1].axvline(best_epoch, color="grey", lw=0.8, ls=":")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("IoU"); ax[1].legend(); ax[1].set_title(CFG["RUN_NAME"])
plt.tight_layout(); plt.savefig(OUT / "curves.png", dpi=130); plt.show()

## 7. Quick test check

Best checkpoint, fixed probability threshold, no TTA. For the reported numbers run
`02_evaluate.ipynb`, which picks the threshold on validation.

In [ ]:
generator.load_weights(str(best_path))
results = {name: quick_eval(ds) for name, ds in
           (("train(%d)" % CFG["TRAIN_EVAL_N"], train_eval_ds), ("val", val_ds), ("test", test_ds))}
with open(OUT / "result.json", "w") as fh:
    json.dump(dict(CFG, best_epoch=best_epoch, minutes=history[-1]["minutes"],
                   **{k + "_buf": v[1] for k, v in results.items()}), fh, indent=2, default=str)
print("%-14s %10s %10s" % ("split", "strict", "buffered"))
for name, (s, b) in results.items():
    print("%-14s %10.4f %10.4f" % (name, s, b))

## 8. Samples (argmax rendered through the palette, plus the road probability)

In [ ]:
n_show = 6
fig, axes = plt.subplots(n_show, 4, figsize=(12, 3 * n_show))
titles = ["satellite", "generated", "target", "road probability"]
for i, (src, tar) in enumerate(test_ds.take(n_show)):
    probs = generator(src, training=False)
    panels = [np.clip(src[0].numpy() * 0.5 + 0.5, 0, 1), seg.render_palette(probs)[0].numpy(),
              np.clip(tar[0].numpy() * 0.5 + 0.5, 0, 1), probs[0, ..., models.ROAD].numpy()]
    for j, im in enumerate(panels):
        axes[i, j].imshow(im, **({"cmap": "magma", "vmin": 0, "vmax": 1} if j == 3 else {}))
        axes[i, j].axis("off")
        if i == 0:
            axes[i, j].set_title(titles[j])
plt.tight_layout(); plt.savefig(OUT / "samples.png", dpi=130); plt.show()
print("outputs in", OUT)